# Agent

## 0 · Setup



In [27]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_portal import GucPortal

portal = GucPortal()   # site defaults to "guc" (set GUC_SITE=giu to switch)
print("logged in to the portal")

logged in to the portal


In [28]:
from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

'ready'

## 1 · The smallest agent that exists

Two arguments — a model and its instructions — and it is already an agent.
No tools yet, so it can talk about grades in general but can't look up
anything real.

In [29]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=llm,
    system_prompt="You are a GUC student's grades assistant. Be brief.",
)

result = agent.invoke({"messages": [
    HumanMessage("What was my GPA in Winter 2024?")
]})
print(result["messages"][-1].content)

I don't have access to your personal grade records or GPA information. To find your Winter 2024 GPA, please:

1. **Log into GUC's student portal** (if available)
2. **Check your transcript** in the registrar's system
3. **Contact the Registrar's Office** directly

Is there anything else I can help you with regarding GPA calculations or grade-related questions?


Notice it can't actually answer — it has no way to look anything up yet. That
is what `tools` fixes in section 3.

## 2 · Everything is messages

`agent.invoke` eats `{"messages": [...]}` and returns the grown transcript in
`result["messages"]`. The transcript IS the context window.

In [30]:
for m in result["messages"]:
    print(m.type.upper().ljust(6), "→", str(m.content)[:100])

HUMAN  → What was my GPA in Winter 2024?
AI     → I don't have access to your personal grade records or GPA information. To find your Winter 2024 GPA,


In [31]:
# continue a conversation: pass the OLD transcript back, grown by one turn
msgs = result["messages"] + [HumanMessage("What term did I just ask about?")]
result2 = agent.invoke({"messages": msgs})
print(result2["messages"][-1].content)

You asked about **Winter 2024**.


### The agent remembers nothing — the list does

A fresh invoke has no past.

In [32]:
fresh = agent.invoke({"messages": [
    HumanMessage("What term did I just ask about?")]})
print(fresh["messages"][-1].content)

You didn't ask about any term yet. This is the start of our conversation. What would you like to know about?


**Your turn** — tell the agent your major or year. Then ask a question that
depends on it: once **continuing the transcript**, once in a **fresh
invoke**. One works, one doesn't — say why in one sentence.

## 3 · Add tools: `tools=[...]`

A tool is a function the model can *read*: name, docstring, typed arguments.
It never executes anything itself — it writes a request, the harness runs it
and feeds the result back.

Rather than redefine them, we pull in the same tools built in
`portal-tools.ipynb`, now extracted into a plain module, `portal_tools.py`
(`list_my_terms`, `list_courses_in_term`, `get_my_grades`,
`list_transcript_years`, `get_my_transcript`), and just `import` them —
logging in to the portal happens once, the first time the module loads.

In [34]:
from portal_tools import (list_my_terms, list_courses_in_term, get_my_grades,
                           list_transcript_years)

ALL_TOOLS = [list_my_terms, list_courses_in_term, get_my_grades,
             list_transcript_years]

INSTRUCTIONS = """You help a GUC student with their grades and transcript.
Use the tools. The portal is slow, so call each tool as few times as you can,
and reuse what you already fetched. Never invent a grade or a term."""

agent = create_agent(model=llm,
    system_prompt=INSTRUCTIONS,
    tools=ALL_TOOLS)   # ← one new argument

result = agent.invoke({"messages": [
    HumanMessage("How did I do in Discrete Math in Winter 2024? Which quiz was weakest?")]})
print(result["messages"][-1].content)

logged in to the portal
Based on your grades in Discrete Math (MATH501) for Winter 2024:

**Overall Performance:** You got **88.54%** in the course.

**Quiz Performance:**
- **Q01:** 8/10
- **Q02:** 5.5/10
- **Q03:** 3/10 ⚠️ **Weakest quiz**

Your weakest quiz was **Q03** where you scored 3 out of 10. There's also a notable drop from Q01 to Q02 to Q03, suggesting the material may have gotten progressively more challenging or you needed more preparation as the quizzes continued.

On the positive side, you did well on all your homework assignments (HW01-03 were perfect at 3/3 each), though HW04 was missed entirely (0/3).


### Read the trace. Always.

The loop just ran for real — model proposed a tool call, the harness ran it,
fed the result back, until the model was ready to answer in plain text.

In [35]:
for m in result["messages"]:
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:120])

HUMAN  → How did I do in Discrete Math in Winter 2024? Which quiz was weakest?
AI     → [{'text': "I'll fetch your grades for Discrete Math in Winter 2024.", 'type': 'text'}, {'id': 'toolu_01TRp4deTLHmFJ7oL4u
TOOL   → {"course": "MET Computer Science 5th Semester - MATH501 Mathematics V (Discrete Math)", "term": "Winter 2024", "items": 
AI     → Based on your grades in Discrete Math (MATH501) for Winter 2024:

**Overall Performance:** You got **88.54%** in the cou


**Your turn** — add a `my_weakest_courses(year: str) -> list` tool that scans
a transcript year for the lowest-graded courses (reuse `get_my_transcript`'s
cache), give it to the agent, and ask *"What were my weakest courses in
2024-2025?"* — read the trace to see which tools it chains.

In [36]:
# your tool here

## 4 · Add structure: `response_format`

The agent works with its tools first, then delivers its final answer in your
schema — no parsing, no extra call.

In [37]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

class GradeReport(BaseModel):
    """A short, structured read on one course's performance."""
    course: str
    term: str
    weakest_assessment: str = Field(description="e.g. 'Q03' or 'HW04'")
    standing: Literal["strong", "on_track", "at_risk"]
    note: str = Field(description="One line for the student")

reporter = create_agent(model=llm,
    system_prompt=INSTRUCTIONS,
    tools=ALL_TOOLS,
    response_format=ToolStrategy(GradeReport))   # ← one new argument

result = reporter.invoke({"messages": [HumanMessage(
    "How am I doing in Discrete Math, Winter 2024?")]})
report = result["structured_response"]
report

GradeReport(course='Discrete Math', term='Winter 2024', weakest_assessment='Q03', standing='on_track', note='Strong overall performance with an 88.54% in the course. Homework is excellent, but recent quizzes show declining performance.')

In [38]:
# it's DATA now — your code can branch on it
if report.standing == "at_risk":
    print("flag for advising:", report.course, "-", report.note)
else:
    print("ok:", report.course)

ok: Discrete Math


**Your turn** — extend `GradeReport` with
`trend: Literal["improving", "flat", "declining"]` computed from the item
sequence, and check the trace: does the agent look at every item before
answering?

In [39]:
# your extended schema here

## 5 · The direct form: `with_structured_output`

Most steps don't need a loop. Turning a free-text question into a
`(term, course)` lookup is a single typed call — no tools, no planning.

In [40]:
class GradeQuery(BaseModel):
    """One grades question, extracted from one message."""
    term: str | None = Field(None, description="e.g. 'Winter 2024'. None if not mentioned.")
    course: str | None = Field(None, description="e.g. 'Discrete Math'. None if not mentioned.")

extractor = llm.with_structured_output(GradeQuery)   # ← the direct form

q = extractor.invoke("how did I do in discrete math last winter?")
print(q)

term='Winter 2024' course='Discrete Math'


### A workflow chains direct calls with your code

No agent needed here: extract → look up the exact term label yourself →
call the portal directly. Fewer, cheaper calls than letting the model loop.

In [41]:
from portal_tools import portal, _seasons


def grades_workflow(question: str) -> str:
    q = extractor.invoke(question)                        # 1 · the direct call
    if not q.term or not q.course:
        return "Need both a term and a course — which term did you mean?"

    term_value = next((v for v, label in _seasons()
                        if q.term.lower() in label.lower()), None)   # 2 · plain code
    if not term_value:
        return f"No term matches {q.term!r}."

    g = portal.get_grades_by_name(q.term, q.course)         # 3 · the real call
    lines = [f"{i.assessment}: {i.grade}" for i in g.items]
    return f"{g.course} ({g.season}):\n" + "\n".join(lines)


print(grades_workflow("how did I do in discrete math last winter?"))

MET Computer Science 5th Semester - MATH501 Mathematics V (Discrete Math) (Winter 2024):
HW01: 3 / 3
HW02: 3 / 3
HW03: 3 / 3
HW04: 0 / 3
Q01: 8 / 10
Q02: 5.5 / 10
Q03: 3 / 10


### The differentiation

| | `with_structured_output` | `create_agent` |
|---|---|---|
| Shape | one typed call | a loop that plans |
| Tools | none — it cannot act | yes — it looks up and acts |
| Control flow | **yours**: order, branches, retries | the model's, until done |
| Typed output | always | add `response_format` |
| Cost | one call | as many as the loop takes |
| Reach for it | **first, by default** | when the path can't be drawn |

> **The rule: use the least powerful abstraction that does the job.**
> `grades_workflow` above (one term, one course) is exactly known in advance
> — direct call. "What were my weakest courses and why?" needs the model to
> decide *which* years and courses to check — that's `create_agent`.

**Your turn** — a `TranscriptQuery` schema (`year`) with the direct form, run
on *"what was my gpa in 2024-2025"*. Then, for a real feature you'd want on
top of this portal, which form — and why? One paragraph, as a comment.

In [42]:
# TranscriptQuery with the direct form + your argument here

---
## Wrap

One call, grown one argument at a time: `create_agent(model, system_prompt)`
→ messages and the proof it remembers nothing → `+ tools` wrapping
`guc_portal` and the visible loop → `+ response_format` and grades as data →
and the differentiation: `with_structured_output` for typed steps you
control, `create_agent` for paths that need planning.